In [0]:
# ============================================================
# GOLD LAYER: Business metrics - FINAL VERSION
# ============================================================
from pyspark.sql.functions import col, to_date, avg, max, min, count, round, current_timestamp

print("📥 Reading silver_telemetry...")
df_silver = spark.read.table("silver_telemetry")

# Extract report date from timestamp for daily aggregation
df_with_date = df_silver.withColumn("report_date", to_date(col("timestamp")))

print("⚙️ Calculating daily summary per device...")
df_gold = df_with_date.groupBy("device_id", "report_date").agg(
    count("event_id").alias("total_events"),
    round(avg("speed_kmh"), 2).alias("avg_speed_kmh"),
    round(avg("battery_pct"), 2).alias("avg_battery_pct"), # key metric for fleet health
    max("engine_temp_c").alias("max_engine_temp_c"),
    min("engine_temp_c").alias("min_engine_temp_c")
)

# Senior touch: order for dashboard readability
df_gold = df_gold.orderBy(col("report_date").desc(), col("device_id"))

# Audit column - when was this Gold table built?
df_gold_final = df_gold.withColumn("gold_processed_at", current_timestamp())

print(f"✅ Gold rows generated: {df_gold_final.count()}")
display(df_gold_final.limit(10))

# Save as managed Delta table - this is what the dashboard will read
print("\n💾 Saving to gold_device_daily_summary...")
df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_device_daily_summary")

print("\n✅ LAKEHOUSE COMPLETE!")
print("   🥉 Bronze: raw_telemetry_dirty.csv -> bronze_telemetry")
print("   🥈 Silver: Cleaned + deduped -> silver_telemetry")
print("   🥇 Gold: Daily aggregated -> gold_device_daily_summary")
print("\nI'll use this Gold table for the Power BI dashboard. It doesn't need raw data anymore.")